# ThinCurr Python Example: Filament Reconstruction Create Jamfit Model

In this notebook we demonstrate how to use the jamfit.py script for creating and preparing for a filament reconstruction

## Load ThinCurr library

In [ ]:
import os 
import sys
import numpy as np
thincurr_python_path = os.getenv('OFT_ROOTPATH')
if thincurr_python_path is not None:
    sys.path.append(os.path.join(thincurr_python_path,'python'))
from OpenFUSIONToolkit._core import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.jamfit import Jamfit
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov, save_sensors, circular_flux_loop

## User Inputs and Creating a Jamfit Object

To create a jamfit object you would need an xml file, a thincurr meshfile, and either the number of threads or an OFT_env object. The xml file can be created using the coils.py file - see the xml example on how to utilize coils.py in order to create an xml file


In [ ]:
myOFT = OFT_env(nthreads=4)
meshfile_thincurr = "thincurr_ex-ports.h5"
xml_name = "oft_in.xml"
jam_obj = Jamfit(xml_name, meshfile_thincurr, oft_env=myOFT)

## Defining Mirnov Sensors and Initializing Within Jamfit

In [ ]:
sensors = []
for i, theta in enumerate(np.linspace(0.0,2.0*np.pi/20.0,3)):
    sensors.append(Mirnov(1.6*np.r_[np.cos(theta),np.sin(theta),0.0], np.r_[0.0,0.0,1.0], 'Bz_{0}'.format(i+1)))
floops_file = save_sensors(sensors)
jam_obj.setup_jamfit(floops_file)

## Running time dependent synthetic simulation 

This allows for jamfit to find the top modes for a given plasma situation 

In [ ]:
dt = 2.E-4
nsteps = 200
time_array = np.array([0, 4E-3, 1.0]).reshape(-1,1) # must be in (n, 1) shape 

coil_currs = np.array([
    [1.E3],
    [1.E3],
    [1.E3]
])

fil_currs = np.array([
    [1.E6],
    [1.E3],
    [0.0]
])

all_currs = np.hstack((time_array, coil_currs, fil_currs))
hist_file = jam_obj.gen_synthetic_data(all_currs, dt, nsteps)

(3, 3)


## Creating a reduced model from the time dependent synthetic run

In [ ]:
num_modes = 10
reduced_filename = "reduced_model.h5"
reduced_model = jam_obj.create_from_runTD_top_modes(num_modes, reduced_filename, all_currs, dt, nsteps) 